# CSI Training on Kaggle (GPU)
Notebook nay dung de train nhanh bang GPU tren Kaggle, khong phu thuoc CPU may local.

## 1) Bat GPU Accelerator tren Kaggle
Vao Settings (goc phai) > Accelerator > chon GPU (T4/P100/A100 neu co).

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('torch') is None:
    print('PyTorch chua co, dang cai...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio'])

import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), 'Chua bat GPU Accelerator trong Kaggle Notebook (Settings > Accelerator > GPU).'

PyTorch chua co, dang cai...
PyTorch: 2.10.0+cpu
CUDA available: False


AssertionError: Chua bat GPU runtime trong Colab. Vao Runtime > Change runtime type > GPU.

## 2) Dua project vao Kaggle working directory
Khuyen nghi upload source code thanh 1 Kaggle Dataset (vi du: pbl5-train-model) roi Add Data vao notebook.
Sau khi xong, project se duoc copy sang /kaggle/working/PBL5-train-model de train va sinh output.

In [ ]:
import os
import shutil

# Doi ten dataset neu ban dat ten khac tren Kaggle
PROJECT_INPUT = '/kaggle/input/pbl5-train-model'
PROJECT_DIR = '/kaggle/working/PBL5-train-model'

assert os.path.isdir(PROJECT_INPUT), (
    f'Khong tim thay {PROJECT_INPUT}. Ban can Add Data truoc (Kaggle Dataset chua source code).'
)

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

shutil.copytree(PROJECT_INPUT, PROJECT_DIR)
os.chdir(PROJECT_DIR)

print('Current dir:', os.getcwd())
print('Root files:', os.listdir('.')[:20])

## 3) Cai dependencies va kiem tra du lieu

In [ ]:
%pip install -q -r requirements.txt

import os
required_files = [
    'data/raw/sit.csv',
    'data/raw/stand.csv',
    'configs/train_default.json',
]
missing = [p for p in required_files if not os.path.exists(p)]
print('Missing files:', missing)
assert not missing, f'Thieu file can thiet: {missing}'
print('Data va config OK, san sang train tren Kaggle GPU.')

## 4) Train CNN2D (khuyen nghi chay cai nay truoc)

In [ ]:
!python -m src.train \
  --config configs/train_default.json \
  --model-type cnn2d \
  --epochs 20 \
  --batch-size 16 \
  --run-name kaggle_cnn2d

## 5) Train LSTM-CNN

In [ ]:
!python -m src.train \
  --config configs/train_default.json \
  --model-type lstmcnn \
  --epochs 30 \
  --batch-size 16 \
  --run-name kaggle_lstmcnn

## 6) Kiem tra checkpoint va log sau train

In [ ]:
!ls -lah models/checkpoints
!ls -lah experiments/results

## 7) (Tuy chon) Dong goi artifact de tai ve tu Kaggle Output

In [ ]:
import os
import shutil

os.makedirs('/kaggle/working/artifacts', exist_ok=True)
shutil.make_archive('/kaggle/working/artifacts/pbl5_results', 'zip', 'experiments/results')
shutil.make_archive('/kaggle/working/artifacts/pbl5_checkpoints', 'zip', 'models/checkpoints')

print('Da tao file zip tai /kaggle/working/artifacts')
print(os.listdir('/kaggle/working/artifacts'))